# InferRules Optimizer (Chapter 6)

This notebook accompanies Chapter 6 §6.5 of *Context Engineering with DSPy*. InferRules walks through your training examples with a teacher LM and asks it to write a numbered list of natural-language rules that distinguish correct from incorrect predictions, then appends those rules to the signature's instructions.

Cost is roughly $0.10–0.30 on the 200-row AI detection task with `gpt-4o-mini` as the task model and `gpt-4o` as the teacher.


## Setup

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import dspy
import pandas as pd
import time
import os
import random
import json
from dotenv import load_dotenv

load_dotenv()

# Pricing per 1M tokens
PRICING = {
    "openai/gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "openai/gpt-4o": {"input": 2.50, "output": 10.00},
}
NUM_THREADS = 16


def calculate_cost(usage: dict) -> float:
    total_cost = 0.0
    for model, data in usage.items():
        if model in PRICING:
            prompt_tokens = data.get("prompt_tokens", 0)
            completion_tokens = data.get("completion_tokens", 0)
            total_cost += (prompt_tokens / 1_000_000) * PRICING[model]["input"]
            total_cost += (completion_tokens / 1_000_000) * PRICING[model]["output"]
    return total_cost


# Initialize models
task_lm = dspy.LM(
    "openai/gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
    max_tokens=16000,
    cache=False,
)

teacher_lm = dspy.LM(
    "openai/gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
    max_tokens=16000,
    cache=False,
)

dspy.configure(lm=task_lm)


# Load dataset
csv_path = '../data/ai_vs_human200.csv'
df = pd.read_csv(csv_path)
examples = df.to_dict(orient='records')
dataset = [dspy.Example(**ex).with_inputs("text") for ex in examples]

random.seed(42)
random.shuffle(dataset)

n = len(dataset)
train_end = int(n * 0.5)
val_end = int(n * 0.75)
trainset = dataset[:train_end]
valset = dataset[train_end:val_end]
testset = dataset[val_end:]

print(f"Train: {len(trainset)}, Val: {len(valset)}, Test: {len(testset)}")


# Define module and metric
class DetectAIText(dspy.Signature):
    """Detects whether text is AI-generated."""
    text: str = dspy.InputField(description="The text to analyze")
    is_ai: bool = dspy.OutputField(description="Whether the text is AI-generated")


class AIDetector(dspy.Module):
    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)


def exact_match(example, response, trace=None):
    return 1 if example.is_ai == response.is_ai else 0


## Run optimizer

In [ ]:
# Run baseline
print("\nRunning baseline...")
detector = AIDetector()
evaluator = dspy.Evaluate(devset=testset, metric=exact_match, num_threads=NUM_THREADS, display_progress=True, display_table=False)
baseline_result = evaluator(detector)
baseline_accuracy = baseline_result.score / 100
print(f"Baseline accuracy: {baseline_accuracy*100:.1f}%")

# Run InferRules
from dspy.teleprompt import InferRules

print("\nRunning InferRules optimizer...")
start_time = time.time()
with dspy.track_usage() as usage:
    optimizer = InferRules(
        metric=exact_match,
        num_candidates=5,
        num_rules=5,
        num_threads=NUM_THREADS,
        teacher_settings={"lm": teacher_lm},
    )
    program_inferrules = optimizer.compile(
        AIDetector(),
        trainset=trainset,
        valset=valset,
    )

    optimized_accuracy = evaluator(program_inferrules).score / 100

elapsed = time.time() - start_time
total_usage = usage.get_total_tokens()

total_tokens = sum(d.get("total_tokens", 0) for d in total_usage.values())
prompt_tokens = sum(d.get("prompt_tokens", 0) for d in total_usage.values())
completion_tokens = sum(d.get("completion_tokens", 0) for d in total_usage.values())
cost = calculate_cost(total_usage)


## Results

In [ ]:
print(f"\n{'='*60}")
print(f"OPTIMIZER: InferRules")
print(f"{'='*60}")
print(f"Optimized Accuracy:  {optimized_accuracy*100:.1f}%")
print(f"Accuracy Uplift:     {(optimized_accuracy - baseline_accuracy)*100:+.1f}%")
print(f"{'-'*60}")
print(f"Total Tokens:        {total_tokens:,}")
print(f"  - Prompt:          {prompt_tokens:,}")
print(f"  - Completion:      {completion_tokens:,}")
print(f"Estimated Cost:      ${cost:.4f}")
print(f"Time Taken:          {elapsed:.1f}s")
print(f"{'='*60}")

# Show the rules
print("\nInferred Rules:")
for pred_name, predictor in program_inferrules.named_predictors():
    if hasattr(predictor, 'signature') and hasattr(predictor.signature, 'instructions'):
        print(predictor.signature.instructions)

# Show demos count
for pred_name, predictor in program_inferrules.named_predictors():
    if hasattr(predictor, 'demos'):
        print(f"\nNumber of demos: {len(predictor.demos)}")

# Save results
results = {
    "baseline_accuracy": baseline_accuracy,
    "optimized_accuracy": optimized_accuracy,
    "accuracy_uplift": optimized_accuracy - baseline_accuracy,
    "total_tokens": total_tokens,
    "prompt_tokens": prompt_tokens,
    "completion_tokens": completion_tokens,
    "cost_usd": cost,
    "time_seconds": elapsed,
    "usage_by_model": {k: dict(v) for k, v in total_usage.items()},
}

with open("inferrules_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nResults saved to inferrules_results.json")
